# Stage 4 — Dense-Sheet Experiment: can smaller/enhanced tiles fix the density collapse?

Focused experiment on ONLY the dense sheets where detection collapsed (recall problem on
100+ symbol sheets). Three configs, same scoring harness, so results are directly comparable:

1. **Molmo2, v3 prompt, normal 1024 tiles** — baseline re-confirm on these sheets
2. **Molmo2, v3 prompt, 512 tiles + 2x upscale + autocontrast** — the "smaller/enhanced" test
3. **Claude (sonnet-4-6), v3 prompt, normal 1024 tiles** — incumbent on the same hard sheets

**Scope caveat (important, per Agent_Pipeline_Facts.md):** the real agent's Stage 3 tiles at
1024x1024. Config 2 deliberately deviates from that. If it wins, the honest finding is "the
substitute needs different tiling than production feeds it" — which would require a Stage 3
change (out of scope) or accepting a benchmark that doesn't match deployment. This experiment
tells us whether the lever *works*, not that we're allowed to use it for free.

**Coordinate handling:** when a tile is upscaled before inference, predictions come back in
upscaled coords and are scaled back down before remapping to sheet coords — otherwise every
box lands in the wrong place.

## 0. Config — which dense sheets, and the tile parameters

In [5]:
# The dense sheets where Molmo2 collapsed (worst F1 + highest symbol density from the
# earlier full run). Trim/extend freely — more sheets = more compute + more Claude API cost.
DENSE_SHEETS = ["151", "216", "233"]

NORMAL_TILE = 1024
NORMAL_OVERLAP = 205
SMALL_TILE = 512
SMALL_OVERLAP = 102   # same ~20% overlap ratio as production
UPSCALE_FACTOR = 2    # config-2 upscales each small tile 2x before inference
RUN_CLAUDE = True     # set False to skip the Claude API config (no key / save cost)

## 1. Fetch the dense sheets from GitHub (no Drive, no upload widget)

The 3 dense sheets + labels are committed in the repo as benchmark fixtures, so we just
`wget` them straight into `/content` — no `drive.mount()`, no file-picker widget (which
doesn't work in the VS Code Colab integration).

In [6]:
from pathlib import Path

SHEETS_DIR = Path("/content/dense_test/sheets"); SHEETS_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR = Path("/content/dense_test/labels"); LABELS_DIR.mkdir(parents=True, exist_ok=True)

BASE = "https://raw.githubusercontent.com/TomGeorge45/pid-opensrc-substitution/main/notebooks/stage4/dense_sheets_sample"
DENSE_SHEETS = ["151", "216", "233"]

for sid in DENSE_SHEETS:
    !wget -q -O {SHEETS_DIR}/{sid}.jpg {BASE}/{sid}.jpg
    !wget -q -O {LABELS_DIR}/{sid}.txt {BASE}/{sid}.txt

print("sheets:", sorted(p.name for p in SHEETS_DIR.glob('*')))
print("labels:", sorted(p.name for p in LABELS_DIR.glob('*')))
# sanity: files should be non-empty (wget -O writes an empty file on 404)
for sid in DENSE_SHEETS:
    assert (SHEETS_DIR/f'{sid}.jpg').stat().st_size > 1000, f'{sid}.jpg missing/empty — check the URL'
    assert (LABELS_DIR/f'{sid}.txt').stat().st_size > 0, f'{sid}.txt missing/empty'
print('\nAll 6 files fetched OK.')

sheets: ['151.jpg', '216.jpg', '233.jpg']
labels: ['151.txt', '216.txt', '233.txt']

All 6 files fetched OK.


## 2. GPU check + install

In [7]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader
!pip install -q transformers==4.57.1 accelerate anthropic

NVIDIA A100-SXM4-80GB, 81920 MiB, 0 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 154.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 57.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 3. Harness — configurable tiling + upscale-aware runner + scoring

In [8]:
import json, re, time
from pathlib import Path
import torch
from PIL import Image, ImageOps

def compute_tile_grid(img_w, img_h, tile_size, overlap):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h); x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            tiles.append({"x0": x0, "y0": y0, "x1": x1, "y1": y1}); x0 += stride
        y0 += stride
    return tiles

def yolo_line_to_xyxy(line, W, H):
    p = line.split(); cx, cy, w, h = (float(v) for v in p[1:5])
    cx, cy, w, h = cx*W, cy*H, w*W, h*H
    return [cx-w/2, cy-h/2, cx+w/2, cy+h/2]

def load_gt_boxes(label_path, W, H):
    if not label_path.exists(): return []
    return [yolo_line_to_xyxy(l, W, H) for l in label_path.read_text().splitlines() if l.strip()]

def _center(b): x0,y0,x1,y1=b; return (x0+x1)/2,(y0+y1)/2
def point_in_box(pt, box): x,y=pt; x0,y0,x1,y1=box; return x0<=x<=x1 and y0<=y<=y1
def pred_point(p): return tuple(p["point"]) if p.get("point") is not None else _center(p["bbox"])

def precision_recall_f1(preds, gt_boxes):
    order = sorted(range(len(preds)), key=lambda i:(preds[i].get("confidence") is None, -(preds[i].get("confidence") or 0)))
    taken = [False]*len(gt_boxes); m=0
    for i in order:
        pt = pred_point(preds[i])
        for j,box in enumerate(gt_boxes):
            if not taken[j] and point_in_box(pt, box): taken[j]=True; m+=1; break
    n_pred, n_gt = len(preds), len(gt_boxes)
    prec = m/n_pred if n_pred else (1.0 if n_gt==0 else 0.0)
    rec = m/n_gt if n_gt else 1.0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
    return {"precision":prec,"recall":rec,"f1":f1,"n_matched":m,"n_pred":n_pred,"n_gt":n_gt}

def stitch_dedup(sheet_preds, dedup_px=30):
    clusters=[]
    for pred in sheet_preds:
        pt=pred_point(pred); et=pred.get("entity_type"); placed=False
        for c in clusters:
            if c["type"]!=et: continue
            cx,cy=pred_point(c["members"][0])
            if ((pt[0]-cx)**2+(pt[1]-cy)**2)**0.5<=dedup_px: c["members"].append(pred); placed=True; break
        if not placed: clusters.append({"type":et,"members":[pred]})
    out=[]
    for c in clusters:
        wc=[m for m in c["members"] if m.get("confidence") is not None]
        out.append(max(wc,key=lambda m:m["confidence"]) if wc else c["members"][0])
    return out

def run_config(sheet_ids, predict_fn, tile_size, overlap, upscale_factor=1, enhance=False, label=""):
    """predict_fn(pil_image) -> (list of detection dicts in that image's coords, parse_err_or_None).
    Upscaled-tile coords are scaled back down before remapping to sheet coords."""
    totals={"n_matched":0,"n_pred":0,"n_gt":0}; parse_fail=0; total_tiles=0
    print(f"\n=== {label} (tile={tile_size}, overlap={overlap}, upscale={upscale_factor}x, enhance={enhance}) ===")
    for sid in sheet_ids:
        img_path=next(SHEETS_DIR.glob(f"{sid}.*")); label_path=LABELS_DIR/f"{sid}.txt"
        sheet=Image.open(img_path).convert("RGB"); W,H=sheet.size
        gt=load_gt_boxes(label_path,W,H)
        sheet_preds=[]
        for t in compute_tile_grid(W,H,tile_size,overlap):
            total_tiles+=1
            crop=sheet.crop((t["x0"],t["y0"],t["x1"],t["y1"]))
            infer_img=crop; scale=1.0
            if upscale_factor and upscale_factor!=1:
                infer_img=crop.resize((crop.width*upscale_factor, crop.height*upscale_factor), Image.LANCZOS)
                scale=1.0/upscale_factor
            if enhance:
                infer_img=ImageOps.autocontrast(infer_img)
            dets,err=predict_fn(infer_img)
            if err: parse_fail+=1; dets=[]
            for d in dets:
                # scale predicted coords back to CROP-local, then offset to SHEET coords
                bx=d["bbox"]; d["bbox"]=[bx[0]*scale+t["x0"], bx[1]*scale+t["y0"], bx[2]*scale+t["x0"], bx[3]*scale+t["y0"]]
                if d.get("point") is not None:
                    d["point"]=[d["point"][0]*scale+t["x0"], d["point"][1]*scale+t["y0"]]
                sheet_preds.append(d)
        deduped=stitch_dedup(sheet_preds)
        r=precision_recall_f1(deduped,gt)
        for k in totals: totals[k]+=r[k]
        print(f"  {sid:6s} gt={r['n_gt']:3d} pred={r['n_pred']:3d} matched={r['n_matched']:3d} P={r['precision']:.2f} R={r['recall']:.2f} F1={r['f1']:.2f}")
    p=totals["n_matched"]/totals["n_pred"] if totals["n_pred"] else 0.0
    rc=totals["n_matched"]/totals["n_gt"] if totals["n_gt"] else 0.0
    f1=2*p*rc/(p+rc) if (p+rc) else 0.0
    print(f"  OVERALL  P={p:.3f} R={rc:.3f} F1={f1:.3f}  | tiles={total_tiles} parse_fail={parse_fail}")
    return {"label":label,"precision":p,"recall":rc,"f1":f1,"tiles":total_tiles,"parse_fail":parse_fail}

## 4. Molmo2 predict_fn (v3 prompt)

In [9]:
from transformers import AutoModelForImageTextToText, AutoProcessor, MaxTimeCriteria, StoppingCriteriaList

molmo_proc = AutoProcessor.from_pretrained("allenai/Molmo2-O-7B", trust_remote_code=True, dtype="auto")
molmo_model = AutoModelForImageTextToText.from_pretrained("allenai/Molmo2-O-7B", trust_remote_code=True, dtype="auto", device_map="cuda")
print("Molmo2 loaded. VRAM:", f"{torch.cuda.memory_allocated()/1e9:.1f} GB")

MOLMO_PROMPT_V3 = ("Point to all P&ID equipment symbols: valves, instrument circles, flanges, "
                   "nozzles, safety devices, and reducers. Point to each one individually, including small ones.")
_PTS = re.compile(r'<(?:points|tracks).*? coords="([0-9\t:;, .]+)"/?>')

def molmo_predict(image):
    messages=[{"role":"user","content":[{"type":"text","text":MOLMO_PROMPT_V3},{"type":"image","image":image}]}]
    inputs=molmo_proc.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_dict=True,return_tensors="pt").to(molmo_model.device)
    with torch.no_grad():
        out=molmo_model.generate(**inputs,max_new_tokens=2048,do_sample=False,
                                  stopping_criteria=StoppingCriteriaList([MaxTimeCriteria(max_time=60.0)]))
    text=molmo_proc.batch_decode(out[:,inputs["input_ids"].shape[1]:],skip_special_tokens=True)[0]
    dets=[]
    for m in _PTS.finditer(text):
        nums=[float(v) for v in re.split(r'[\t:;, ]+',m.group(1).strip()) if v]
        if len(nums)%3!=0:
            if len(nums)>=2 and nums[0]==nums[1] and (len(nums)-1)%3==0: nums=nums[1:]
            else: return None,"coords not mult of 3"
        for i in range(0,len(nums),3):
            _f,xs,ys=nums[i:i+3]; x,y=xs/1000*image.width,ys/1000*image.height
            dets.append({"bbox":[x-20,y-20,x+20,y+20],"confidence":None,"entity_type":None,"point":[x,y]})
    if not dets and "<point" in text.lower(): return None,"point tags but no match"
    return dets,None

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


processor_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

processing_molmo2.py: 0.00B [00:00, ?B/s]

video_processing_molmo2.py: 0.00B [00:00, ?B/s]

...

model-00001-of-00007.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00003-of-00007.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

Molmo2 loaded. VRAM: 31.0 GB


## 5. Config 1 — Molmo2 baseline (1024 tiles) & Config 2 — Molmo2 small+enhanced (512, 2x, autocontrast)

In [ ]:
results = []
results.append(run_config(DENSE_SHEETS, molmo_predict, NORMAL_TILE, NORMAL_OVERLAP,
                          upscale_factor=1, enhance=False, label="Molmo2 / 1024 baseline"))
results.append(run_config(DENSE_SHEETS, molmo_predict, SMALL_TILE, SMALL_OVERLAP,
                          upscale_factor=UPSCALE_FACTOR, enhance=True, label="Molmo2 / 512 + 2x + autocontrast"))


=== Molmo2 / 1024 baseline (tile=1024, overlap=205, upscale=1x, enhance=False) ===


## 6. Claude predict_fn (v3 prompt) + Config 3 — Claude on the same dense sheets, 1024 tiles

In [ ]:
if RUN_CLAUDE:
    import os, base64
    from io import BytesIO
    import anthropic
    os.environ["ANTHROPIC_API_KEY"] = "paste-your-anthropic-api-key-here"  # <-- edit; never commit
    claude_client = anthropic.Anthropic()
    CLAUDE_MODEL = "claude-sonnet-4-6"

    CLAUDE_PROMPT_V3 = """You are an expert P&ID reviewer analyzing one tile cropped from a larger Piping & Instrumentation Diagram. The image is {W}x{H} pixels.

First, briefly scan the tile region by region (top-left, top-right, center, bottom-left, bottom-right) and note what equipment symbols you see in each region. Dense tiles can contain 50+ symbols - keep going until every region is covered.

Count as a symbol: valves, instrument circles/bubbles, flanges, nozzles, safety devices, reducers, and other discrete equipment icons.
Not symbols: plain pipe/line segments, text labels alone, dimension arrows, borders.

Then output your final answer as a JSON array of arrays inside a ```json code fence. Each inner array is exactly: [x0, y0, x1, y1, confidence, "entity_type"] - tight boxes around just the icon, absolute pixel coordinates (top-left origin), confidence 0.0-1.0. If no symbols: []"""

    def claude_predict(image):
        buf=BytesIO(); image.save(buf,format="PNG")
        b64=base64.b64encode(buf.getvalue()).decode()
        resp=claude_client.messages.create(model=CLAUDE_MODEL,max_tokens=8000,messages=[{"role":"user","content":[
            {"type":"image","source":{"type":"base64","media_type":"image/png","data":b64}},
            {"type":"text","text":CLAUDE_PROMPT_V3.format(W=image.width,H=image.height)}]}])
        text=resp.content[0].text
        fenced=re.findall(r'```(?:json)?\s*(\[[\s\S]*?\])\s*```',text)
        cleaned=fenced[-1].strip() if fenced else re.sub(r'\s*```$','',re.sub(r'^```(?:json)?\s*','',text.strip()))
        try: data=json.loads(cleaned)
        except json.JSONDecodeError as e: return None,f"JSONDecodeError: {e}"
        if not isinstance(data,list): return None,"not a list"
        dets=[]
        for it in data:
            if not (isinstance(it,list) and len(it)==6): return None,f"malformed: {it}"
            x0,y0,x1,y1,c,et=it
            dets.append({"bbox":[float(x0),float(y0),float(x1),float(y1)],"confidence":float(c),"entity_type":str(et)})
        return dets,None

    results.append(run_config(DENSE_SHEETS, claude_predict, NORMAL_TILE, NORMAL_OVERLAP,
                              upscale_factor=1, enhance=False, label="Claude / 1024 baseline"))

## 7. Summary comparison

In [ ]:
print(f"{'Config':40s} {'P':>6s} {'R':>6s} {'F1':>6s} {'tiles':>6s}")
for r in results:
    print(f"{r['label']:40s} {r['precision']:6.3f} {r['recall']:6.3f} {r['f1']:6.3f} {r['tiles']:6d}")
print("\nDense sheets:", DENSE_SHEETS)
print("Reference (full-20 run): Molmo2 v1 F1=0.434, Claude v1 F1=0.380 — but this is the "
      "dense subset only, where both scored far lower, so compare configs to each OTHER here.")